<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/oscillatory_mmnn_particle_parameter_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oscillatory MMNN-DTB with direct particle and parameter updates

This notebook isolates the first update approach. It keeps fixed labels \(z_i\), a physical particle state \(X_k(z_i)\), and MMNN parameters \(\theta_k\). The MMNN is used only to generate the tangent basis; no callable accumulated-map object and no separate nonlinear-network trajectory are constructed.

Given \(X_{k-1}\) and \(\theta_{k-1}\), compute

\[
J_i^{k-1}=D_\theta T_{\theta_{k-1}}(z_i),
\qquad
\alpha_k=\arg\min_\alpha\sum_i\left\|J_i^{k-1}\alpha-b(X_{k-1}(z_i))\right\|^2,
\]

then update only the two requested states:

\[
X_k(z_i)=X_{k-1}(z_i)+hJ_i^{k-1}\alpha_k,
\qquad
\theta_k=\theta_{k-1}+h\alpha_k.
\]

The residual MMNN starts from \(T_{\theta_0}(z)=z\). Its random \(W,b\) feature parameters remain frozen, while its trainable \(A,c\) coefficients define the DTB parameter vector.

In [ ]:
# BLOCK 0 — Colab/repository setup and experiment controls.
# Change values only in the configuration section when starting a sweep.
from pathlib import Path
import os, subprocess, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO = Path("/content/dtb-colab-experiments")
BRANCH = "codex/game-dynamics-dtb"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/sun-mengwei/dtb-colab-experiments.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"], check=True)
EXPERIMENT_DIR = REPO / "DTB_Game_Ver2"
sys.path.insert(0, str(EXPERIMENT_DIR))
sys.path.insert(0, str(REPO / "DTB_Game_Ver1"))
os.chdir(EXPERIMENT_DIR)

try:
    import torch._dynamo.compiled_autograd
except (AttributeError, ImportError):
    pass
from dtb import device, flat_params, write_flat_into_model
from network import MMNN
from run_game_dtb import game_dtb_basis_matrices, map_at
from game_dtb.projection import truncated_svd_solve

# --------------------------- configuration ---------------------------
SEED = 2026
N = 1000
T, H = 1.0, 0.005
K = round(T / H)
MMNN_WIDTH, MMNN_RANK, MMNN_DEPTH = 12, 12, 3
SVD_RTOL = 1e-3  # filters parameter directions below rtol * sigma_max
JACOBIAN_CHUNK = 128
DOMAIN_LOW, DOMAIN_HIGH = -1.0, 1.0
DIRECTION_NORM_TOL = 1e-8
LAMBDA, GAMMA = 0.5, 0.2
EPSILON, OMEGA = 0.5, 4 * math.pi
SNAPSHOT_STEPS = sorted(set([0, K // 4, K // 2, 3 * K // 4, K]))

DEVICE = device()
DTYPE = torch.float32
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
print({"device": str(DEVICE), "particles": N, "steps": K, "h": H,
       "svd_rtol": SVD_RTOL, "architecture": "MMNN",
       "width": MMNN_WIDTH, "rank": MMNN_RANK, "depth": MMNN_DEPTH})

In [ ]:
# BLOCK 1 — Define the exact oscillatory deterministic game.
def phi(x):
    """Potential used for phase-plane contours and energy diagnostics."""
    x1, x2 = x.unbind(-1)
    return (
        -0.5 * LAMBDA * (x1.square() + x2.square())
        -0.5 * GAMMA * (x1 - x2).square()
        +(EPSILON / OMEGA) * (torch.cos(OMEGA * x1) + torch.cos(OMEGA * x2))
    )

def b(x):
    """Simultaneous payoff-gradient velocity b(x)=grad Phi(x)."""
    x1, x2 = x.unbind(-1)
    return torch.stack((
        -LAMBDA * x1 - GAMMA * (x1 - x2) - EPSILON * torch.sin(OMEGA * x1),
        -LAMBDA * x2 - GAMMA * (x2 - x1) - EPSILON * torch.sin(OMEGA * x2),
    ), dim=-1)

# Exact implementation check: b must equal grad Phi.
check_x = torch.tensor([[0.17, -0.31]], device=DEVICE, dtype=DTYPE, requires_grad=True)
assert (torch.autograd.grad(phi(check_x).sum(), check_x)[0] - b(check_x)).abs().max() < 2e-6

In [ ]:
# BLOCK 2 — Define a residual MMNN and initialize X_0=T_{theta_0}=identity.
# Frozen MMNN features remain fixed; DTB evolves only trainable A,c coefficients.
class ResidualMMNNMap(torch.nn.Module):
    """T_theta(z)=z+F_theta(z), with F_theta implemented by an MMNN."""
    def __init__(self, dim, width, rank, depth, activation="tanh",
                 dtype=torch.float32):
        super().__init__()
        self.net = MMNN(d_in=dim, width=width, rank=rank, depth=depth,
                        d_out=dim, activation=activation, dtype=dtype)
        # Start from the identity without changing the MMNN feature draws.
        final_layer = self.net.layers[-1]
        torch.nn.init.zeros_(final_layer.A)
        torch.nn.init.zeros_(final_layer.c)

    def forward(self, z):
        return z + self.net(z)

# The labels z_i never change; X_k and theta_k evolve separately afterward.
z = (DOMAIN_LOW + (DOMAIN_HIGH - DOMAIN_LOW)
     * torch.rand(N, 2, dtype=DTYPE)).to(DEVICE)
model = ResidualMMNNMap(dim=2, width=MMNN_WIDTH, rank=MMNN_RANK,
                        depth=MMNN_DEPTH, activation="tanh", dtype=DTYPE).to(DEVICE)
theta_0, structure, _ = flat_params(model)
theta_0 = theta_0.to(DEVICE)
selected = torch.arange(theta_0.numel(), device=DEVICE)  # evolve every parameter
x0 = map_at(theta_0, z, model, structure).detach()
assert torch.allclose(x0, z, atol=1e-7, rtol=0), "residual MMNN must start at identity"
frozen_parameters = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"trainable parameters: {theta_0.numel()} (all used in the SVD solve); "
      f"frozen MMNN feature parameters: {frozen_parameters}")

In [ ]:
# BLOCK 3 — Apply only the direct particle and parameter updates.
def particle_direction_cosine(projected_velocity, target_velocity, tol=DIRECTION_NORM_TOL):
    """Cosine alignment for each particle; near-zero vectors return NaN."""
    projected_norm = torch.linalg.vector_norm(projected_velocity, dim=1)
    target_norm = torch.linalg.vector_norm(target_velocity, dim=1)
    valid = (projected_norm > tol) & (target_norm > tol)
    cosine = torch.full_like(projected_norm, float("nan"))
    cosine[valid] = (
        (projected_velocity[valid] * target_velocity[valid]).sum(1)
        / (projected_norm[valid] * target_norm[valid])
    ).clamp(-1, 1)
    return cosine

def run_particle_parameter_dtb(theta_initial, labels):
    """Evolve X_k and theta_k with the same projected coefficient alpha_k."""
    theta_previous = theta_initial.detach().clone()
    x_previous = map_at(theta_previous, labels, model, structure).detach()
    history = {name: [] for name in (
        "trajectory", "projection_residual", "retained_rank",
        "sigma_max", "sigma_min_retained", "alpha_norm",
        "fraction_negative", "fraction_above_half",
    )}
    history["trajectory"].append(x_previous.cpu())
    tic = time.perf_counter()

    for step in range(1, K + 1):
        # J_{theta_{k-1}} is evaluated at the fixed labels z_i.
        _, jacobians, stacked_jacobian = game_dtb_basis_matrices(
            theta_previous, selected, labels, model, structure,
            chunk=JACOBIAN_CHUNK
        )

        # alpha_k fits the game velocity at the current physical state X_{k-1}.
        target_velocity = b(x_previous)
        alpha_k, svd = truncated_svd_solve(
            stacked_jacobian, target_velocity.reshape(-1), rtol=SVD_RTOL
        )
        tangent_velocity = torch.einsum("ndm,m->nd", jacobians, alpha_k)
        cosine = particle_direction_cosine(tangent_velocity, target_velocity)
        valid = torch.isfinite(cosine)

        # The only two state updates considered in this notebook.
        x_k = x_previous + H * tangent_velocity.detach()
        theta_k = theta_previous + H * alpha_k

        history["trajectory"].append(x_k.cpu())
        history["projection_residual"].append(svd.relative_residual)
        history["retained_rank"].append(svd.retained_rank)
        history["sigma_max"].append(svd.sigma_max)
        history["sigma_min_retained"].append(svd.sigma_min_retained)
        history["alpha_norm"].append(float(torch.linalg.norm(alpha_k)))
        history["fraction_negative"].append(float((cosine[valid] < 0).float().mean()))
        history["fraction_above_half"].append(float((cosine[valid] > 0.5).float().mean()))

        x_previous = x_k.detach()
        theta_previous = theta_k.detach()
        if step % max(1, K // 5) == 0:
            print(f"{step:3d}/{K}: residual={svd.relative_residual:.2e}, "
                  f"rank={svd.retained_rank}, |alpha|={float(alpha_k.norm()):.2e}")

    write_flat_into_model(model, theta_previous, structure)
    history["trajectory"] = torch.stack(history["trajectory"])
    for key in history:
        if key != "trajectory":
            history[key] = np.asarray(history[key])
    history["wall_seconds"] = time.perf_counter() - tic
    return theta_previous, history

In [ ]:
# BLOCK 4 — Compare the requested update with paired explicit Euler.
# Both trajectories use the same initial particles, step size, and game velocity.
x_reference = z.detach().clone()
reference_trajectory = [x_reference.cpu()]
for _ in range(K):
    x_reference = x_reference + H * b(x_reference)
    reference_trajectory.append(x_reference.detach().cpu())
reference_trajectory = torch.stack(reference_trajectory)

theta_K, result = run_particle_parameter_dtb(theta_0, z)
assert torch.isfinite(theta_K).all() and torch.isfinite(result["trajectory"]).all()
assert result["trajectory"].shape == reference_trajectory.shape

trajectory_rmse = torch.sqrt(torch.mean(torch.sum(
    (result["trajectory"] - reference_trajectory).square(), dim=-1
), dim=1))
reference_rms = torch.sqrt(torch.mean(torch.sum(
    reference_trajectory.square(), dim=-1
), dim=1))
relative_rmse = trajectory_rmse / (reference_rms + 1e-12)
potential_reference = phi(reference_trajectory).mean(1)
potential_dtb = phi(result["trajectory"]).mean(1)

print(f"wall time: {result['wall_seconds']:.1f} s")
print(f"mean/max residual: {result['projection_residual'].mean():.3e} / "
      f"{result['projection_residual'].max():.3e}")
print(f"final particle RMSE: {trajectory_rmse[-1]:.3e}; "
      f"relative {relative_rmse[-1]:.3e}")

In [ ]:
# BLOCK 5 — Plot only X_k, the Euler reference, and projection diagnostics.
grid_axis = torch.linspace(-1.05, 1.05, 140)
grid_x1, grid_x2 = torch.meshgrid(grid_axis, grid_axis, indexing="xy")
potential_grid = phi(
    torch.stack((grid_x1.ravel(), grid_x2.ravel()), -1)
).reshape(grid_x1.shape)

fig, axes = plt.subplots(2, len(SNAPSHOT_STEPS),
                         figsize=(3 * len(SNAPSHOT_STEPS), 6),
                         sharex=True, sharey=True)
for column, step in enumerate(SNAPSHOT_STEPS):
    for row in range(2):
        axes[row, column].contour(
            grid_x1, grid_x2, potential_grid, levels=16,
            linewidths=.45, alpha=.55
        )
    dtb_points = result["trajectory"][step].numpy()
    reference_points = reference_trajectory[step].numpy()
    axes[0, column].scatter(dtb_points[:, 0], dtb_points[:, 1], s=8, alpha=.55)
    axes[1, column].scatter(reference_points[:, 0], reference_points[:, 1],
                            s=8, alpha=.55)
    axes[0, column].set_title(f"t={step * H:.2f}")
    for row in range(2):
        axes[row, column].grid(alpha=.2)
        axes[row, column].set_aspect("equal")
axes[0, 0].set_ylabel(r"particle update $X_k$")
axes[1, 0].set_ylabel("explicit Euler")
for axis in axes[1]:
    axis.set_xlabel(r"$x_1$")
plt.tight_layout(); plt.show()

state_times = np.arange(K + 1) * H
step_times = np.arange(K) * H
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
axes[0, 0].semilogy(state_times, trajectory_rmse + 1e-15, label="absolute")
axes[0, 0].semilogy(state_times, relative_rmse + 1e-15, label="relative")
axes[0, 0].set_title("particle error against Euler"); axes[0, 0].legend()
axes[0, 1].semilogy(step_times, result["projection_residual"] + 1e-15)
axes[0, 1].set_title(r"$\|J\alpha-b(X)\|/\|b(X)\|$")
axes[0, 2].plot(step_times, result["retained_rank"])
axes[0, 2].set_title("retained tangent rank")
axes[1, 0].semilogy(step_times, result["alpha_norm"] + 1e-15)
axes[1, 0].set_title(r"$\|\alpha_k\|_2$")
axes[1, 1].semilogy(step_times, result["sigma_min_retained"] + 1e-15)
axes[1, 1].set_title(r"$\sigma_{\min,\mathrm{retained}}(J)$")
axes[1, 2].plot(step_times, result["fraction_negative"],
                label=r"fraction $c_i<0$")
axes[1, 2].plot(step_times, result["fraction_above_half"],
                label=r"fraction $c_i>0.5$")
axes[1, 2].set_title("local direction alignment"); axes[1, 2].legend()
for axis in axes.ravel():
    axis.set_xlabel("time")
    axis.grid(alpha=.25)
plt.tight_layout(); plt.show()

fig, axis = plt.subplots(figsize=(6, 4))
axis.plot(state_times, potential_reference, label="Euler")
axis.plot(state_times, potential_dtb, label=r"particle update $X_k$")
axis.set(xlabel="time", ylabel="mean potential", title="Mean potential")
axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.show()

## Reading this first update experiment

- `trajectory[k]` is the cached value of the physical state \(X_k\) on the fixed initial labels.
- The MMNN supplies \(J_{\theta_{k-1}}\), but this notebook does not replace \(X_k\) by \(T_{\theta_k}(z)\) after the update.
- There is no `AccumulatedMovingTangentMap`: the experiment investigates only the recursive particle update and the simultaneous parameter update.
- `projection_residual` measures how well the current MMNN tangent space represents \(b(X_{k-1})\).
- The explicit-Euler reference uses the same \(h\), isolating the error introduced by tangent projection.
- The random MMNN \(W,b\) features stay frozen; only trainable \(A,c\) parameters evolve through \(\theta_k=\theta_{k-1}+h\alpha_k\).